In [ ]:
%load_ext IPython.extensions.autoreload
%autoreload 2

import pathlib
from pathlib import Path
import sys
# Add src to path for importing local modules
sys.path.insert(0, str(Path("../src").resolve()))
from pickle_compat import enable_dill_pathlib_compat
enable_dill_pathlib_compat()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import dill
from scipy import stats
from scipy.ndimage import gaussian_filter1d
from scipy.spatial.distance import cdist
from sklearn.manifold import MDS
from trompy import save_figure_atomic
from extract_behav_parameters import smooth_array


from figure_config import (
    configure_matplotlib, COLORS, HEATMAP_CMAP_DIV,
    HEATMAP_CMAP_RED, HEATMAP_CMAP_BLUE,
    DATAFOLDER, RESULTSFOLDER, FIGSFOLDER,
    HEATMAP_VLIM_BEHAV, YLIMS_BEHAV,
    BEHAV_SMOOTH_WINDOW, SAVE_FIGS
)
from figure_plotting import (
    smooth_array, get_heatmap_data_by_rat, get_mean_snips, get_auc,
    init_heatmap_figure, init_snips_figure, make_heatmap,
    plot_snips, plot_auc_summary, print_auc_stats, get_trial_data_by_rat,
    scale_vlim_to_data, calculate_ylims,
    draw_regression_line, make_correlation_plot_simba
)

# Configure matplotlib
configure_matplotlib()
colors = COLORS  # Use shared color palette
custom_cmap_red = HEATMAP_CMAP_RED  # Use shared colormap
custom_cmap_blue = HEATMAP_CMAP_BLUE  # Use shared colormap

custom_cmap = HEATMAP_CMAP_DIV  # Use shared diverging colormap

SAVE_FIGS = True

In [ ]:
assembled_data_path = DATAFOLDER / "assembled_data.pickle"

with open(assembled_data_path, "rb") as f:
    data = dill.load(f)

# Extract main components
x_array = data["x_array"]
snips_photo = data["snips_photo"]
snips_simba = data["snips_simba"]

params = data.get("params", {})
metadata = data.get("metadata", {})

print(f"Loaded assembled data from {assembled_data_path}")
print(f"\nData structure:")
print(f"  - x_array shape: {x_array.shape}")
print(f"  - snips_photo shape: {snips_photo.shape}")
print(f"  - snips_simba shape: {snips_simba.shape}")
print(f"  - x_array columns: {x_array.columns.tolist()}")
print(f"  - Number of trials: {len(x_array)}")


In [ ]:
snips_to_plot = np.array(smooth_array(snips_simba))
vlim_behav = (-1, 1)
vlim_photo = (-1, 1)
rats = x_array.query("infusiontype == '45NaCl'").id.unique()


def make_rep_rat_plot(snips_simba, snips_photo, x_array, rat):
    f, ax = plt.subplots(ncols=4, figsize=(7, 1.5),
                        gridspec_kw={"wspace": 0.6, "bottom": 0.35,
                                     "left": 0.05, "right": 0.95},)

    behav_data = snips_simba[x_array.query("id == @rat and condition == 'deplete'").index, :]
    photo_data = snips_photo[x_array.query("id == @rat and condition == 'deplete'").index, :]

    behav_auc = x_array.query("id == @rat and condition == 'deplete'").auc_simba.values


    sns.heatmap(behav_data, ax=ax[0], cmap=custom_cmap,
                vmin=vlim_behav[0], vmax=vlim_behav[1],
                cbar=False
                )
    ax[0].set_yticks([])
    ax[0].set_xticks([])

    ax[1].plot(np.mean(behav_data[:,50:150], axis=1), color=colors[3])
    ax[1].set_xlabel("Trial")
    ax[1].set_ylabel("AUC")
    sns.despine(ax=ax[1], offset=5)

    sns.heatmap(photo_data, ax=ax[2], cmap=custom_cmap,
                vmin=vlim_photo[0], vmax=vlim_photo[1],
                cbar=False
                )


    ax[3].plot(np.mean(photo_data[:,50:150], axis=1), color=colors[3])



    for axis in [ax[0], ax[2]]:
        axis.set_yticks([])
        axis.set_xticks([])
        axis.set_ylabel("Trials")
        axis.plot((149, 198), (51, 51), color="k", linewidth=2, alpha=0.3, clip_on=False)
        axis.text(175, 53, "5 s", ha="center", va="top", fontsize=10, color="k")

    for axis in [ax[1], ax[3]]:
        axis.axhline(0, color="gray", linestyle="--", linewidth=1)
        axis.set_xlabel("Trial")
        axis.set_ylabel("AUC")
        sns.despine(ax=axis, offset=5)

    return f


In [ ]:
# possible rats 3, 7, 8, 0


f = make_rep_rat_plot(snips_simba, snips_photo, x_array, rats[0])

if SAVE_FIGS:
    save_figure_atomic(f, "figure_S3a_individual_rats", FIGSFOLDER)


f = make_rep_rat_plot(snips_simba, snips_photo, x_array, rats[7])

if SAVE_FIGS:
    save_figure_atomic(f, "figure_S3b_individual_rats", FIGSFOLDER)

f = make_rep_rat_plot(snips_simba, snips_photo, x_array, rats[8])

if SAVE_FIGS:
    save_figure_atomic(f, "figure_S3c_individual_rats", FIGSFOLDER)

f = make_rep_rat_plot(snips_simba, snips_photo, x_array, rats[3])

if SAVE_FIGS:
    save_figure_atomic(f, "figure_S3d_individual_rats", FIGSFOLDER)

f = make_rep_rat_plot(snips_simba, snips_photo, x_array, rats[2])

if SAVE_FIGS:
    save_figure_atomic(f, "figure_S3e_individual_rats", FIGSFOLDER)

In [ ]:
for rat in rats:
    f = make_rep_rat_plot(snips_simba, snips_photo, x_array, rat)